In [3]:
from abc import ABC, abstractmethod
from pathlib import Path
from typing import Iterator
import pandas as pd
CHUNK_SIZE = 10_000
class DataReader(ABC):
    """Abstract interface for supported data readers."""
    @abstractmethod
    def read_chunks(self, file_path: str) -> Iterator[pd.DataFrame]:
        """Read a data source in manageable chunks."""
        raise NotImplementedError
class CSVReader(DataReader):
    """Read CSV files in chunks."""
    def read_chunks(self, file_path: str) -> Iterator[pd.DataFrame]:
        path = Path(file_path)
        if not path.is_file():
            raise FileNotFoundError(f"Dataset not found: {path}")
        if path.suffix.lower() != ".csv":
            raise ValueError("CSVReader supports CSV files only.")
        try:
            yield from pd.read_csv(path, chunksize=CHUNK_SIZE)
        except pd.errors.EmptyDataError as error:
            raise ValueError(
                "The selected CSV is empty or has no header row."
            ) from error
        except pd.errors.ParserError as error:
            raise ValueError(
                f"The selected CSV is malformed: {error}"
            ) from error

        except UnicodeDecodeError as error:
            raise ValueError(
                "The selected CSV is not readable as UTF-8 text."
            ) from error
class LogReader(DataReader):
    """Read log files in chunks."""
    def read_chunks(self, file_path: str) -> Iterator[pd.DataFrame]:
        path = Path(file_path)
        if not path.is_file():
            raise FileNotFoundError(f"Log file not found: {path}")
        if path.suffix.lower() != ".log":
            raise ValueError("LogReader supports .log files only.")

        try:
            records = []

            with path.open("r", encoding="utf-8") as file:
                for line in file:
                    records.append({"log_message": line.rstrip("\n")})

                    if len(records) >= CHUNK_SIZE:
                        yield pd.DataFrame(records)
                        records = []

            if records:
                yield pd.DataFrame(records)

        except UnicodeDecodeError as error:
            raise ValueError(
                "The selected log file is not readable as UTF-8 text."
            ) from error


class DataSourceFactory:
    """Create a suitable reader based on the input file extension."""

    @staticmethod
    def create_reader(file_path: str) -> DataReader:
        extension = Path(file_path).suffix.lower()

        if extension == ".csv":
            return CSVReader()

        if extension == ".log":
            return LogReader()

        raise ValueError(
            f"Unsupported file format: {extension}. "
            "Supported formats are .csv and .log."
        )


In [6]:
from dataclasses import dataclass, field
import pandas as pd
@dataclass
class ValidationResult:
    """Store validation outcomes for a data chunk."""
    valid_records: pd.DataFrame
    invalid_records: pd.DataFrame
    missing_cells: int
    duplicate_records: int
    issues: list[str] = field(default_factory=list)
class DataValidator:
    """Validate retail transaction data."""
    REQUIRED_COLUMNS = [
        "InvoiceNo",
        "StockCode",
        "Description",
        "Quantity",
        "InvoiceDate",
        "UnitPrice",
        "CustomerID",
        "Country",
    ]
    def validate(self, data: pd.DataFrame) -> ValidationResult:
        """Validate a chunk and separate valid and invalid records.""",
        if data.empty:
            return ValidationResult(
                valid_records=data.copy(),
                invalid_records=data.copy(),
                missing_cells=0,
                duplicate_records=0,
                issues=["The input data chunk is empty."],
            )
        df = data.copy()
        issues = []
        # Check required columns
        missing_columns = [
            column
            for column in self.REQUIRED_COLUMNS
            if column not in df.columns
        ]
        if missing_columns:
            raise ValueError(
                f"Missing required columns: {missing_columns}"
            )
        # Count missing cells
        missing_cells = int(df.isna().sum().sum())

        # Identify duplicate records
        duplicate_mask = df.duplicated(keep="first")
        duplicate_records = int(duplicate_mask.sum())

        if duplicate_records:
            issues.append(
                f"Found {duplicate_records} duplicate records."
            )

        # Convert numeric columns
        df["Quantity"] = pd.to_numeric(
            df["Quantity"], errors="coerce"
        )
        df["UnitPrice"] = pd.to_numeric(
            df["UnitPrice"], errors="coerce"
        )
        # Validate required values and numeric fields
        invalid_mask = (
            df[self.REQUIRED_COLUMNS].isna().any(axis=1)
            | df["Quantity"].isna()
            | df["UnitPrice"].isna()
            | ~df["Quantity"].map(
                lambda value: pd.notna(value)
                and float(value).is_integer()
            )
            | (df["UnitPrice"] < 0)
        )
        # Treat duplicates as invalid for this validation
        invalid_mask = invalid_mask | duplicate_mask

        valid_records = df.loc[~invalid_mask].copy()
        invalid_records = df.loc[invalid_mask].copy()

        if not invalid_records.empty:
            issues.append(
                f"Found {len(invalid_records)} invalid records."
            )

        return ValidationResult(
            valid_records=valid_records,
            invalid_records=invalid_records,
            missing_cells=missing_cells,
            duplicate_records=duplicate_records,
            issues=issues,
        )


In [8]:
import pandas as pd
class CleanTextStage:
    """Clean and standardise text values in a DataFrame."""
    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.copy()
        text_columns = df.select_dtypes(
            include=["object", "string"]
        ).columns
        for column in text_columns:
            df[column] = (
                df[column]
                .astype("string")
                .str.strip()
            )

        if "Country" in df.columns:
            df["Country"] = df["Country"].str.title()

        if "Description" in df.columns:
            df["Description"] = (
                df["Description"]
                .str.replace(r"\s+", " ", regex=True)
            )
        return df

In [9]:
import pandas as pd
class FeatureEngineeringStage:
    """Create useful features from retail transaction data."""
    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.copy()
        if "InvoiceDate" in df.columns:
            df["InvoiceDate"] = pd.to_datetime(
                df["InvoiceDate"],
                errors="coerce"
            )
            df["Year"] = df["InvoiceDate"].dt.year
            df["Month"] = df["InvoiceDate"].dt.month
            df["Day"] = df["InvoiceDate"].dt.day
            df["Hour"] = df["InvoiceDate"].dt.hour
            df["DayOfWeek"] = (
                df["InvoiceDate"].dt.day_name()
            )
        if "CustomerID" in df.columns:
            df["CustomerID"] = df["CustomerID"].astype(
                "string"
            ).str.strip()
        return df

In [10]:
import pandas as pd
class RevenueFeatureStage:
    """Generate revenue-related features."""
    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.copy()
        required_columns = ["Quantity", "UnitPrice"]
        missing_columns = [
            column
            for column in required_columns
            if column not in df.columns
        ]
        if missing_columns:
            raise ValueError(
                f"Missing required columns: {missing_columns}"
            )
        df["Quantity"] = pd.to_numeric(
            df["Quantity"], errors="coerce"
        )
        df["UnitPrice"] = pd.to_numeric(
            df["UnitPrice"], errors="coerce"
        )
        df["Revenue"] = df["Quantity"] * df["UnitPrice"]
        return df


In [12]:
import pandas as pd
class RevenueFeatureStage:
    """Generate revenue-related features."""
    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.copy()
        required_columns = ["Quantity", "UnitPrice"]
        missing_columns = [
            column
            for column in required_columns
            if column not in df.columns
        ]
        if missing_columns:
            raise ValueError(
                f"Missing required columns: {missing_columns}"
            )
        df["Quantity"] = pd.to_numeric(
            df["Quantity"], errors="coerce"
        )
        df["UnitPrice"] = pd.to_numeric(
            df["UnitPrice"], errors="coerce"
        )
        df["Revenue"] = df["Quantity"] * df["UnitPrice"]
        return df

In [13]:
import pandas as pd
class RetailTransformationPipeline:
    """Execute the retail data transformation stages."""
    def __init__(self):
        self.stages = [
            CleanTextStage(),
            FeatureEngineeringStage(),
            RevenueFeatureStage(),
        ]
    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        result = data.copy()

        for stage in self.stages:
            result = stage.transform(result)
        return result
def create_retail_pipeline() -> RetailTransformationPipeline:
    """Create and return the retail transformation pipeline."""
    return RetailTransformationPipeline()

In [16]:
from abc import ABC, abstractmethod
import pandas as pd
class AnalyticsStrategy(ABC):
    """Common interface for retail analytics strategies."""
    @abstractmethod
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        """Perform an analytical operation on the dataset."""
        raise NotImplementedError
class PeakSalesStrategy(AnalyticsStrategy):
    """Analyse transaction activity by hour."""
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.copy()
        df["InvoiceDate"] = pd.to_datetime(
            df["InvoiceDate"], errors="coerce"
        )
        df = df.dropna(subset=["InvoiceDate"])
        df["Hour"] = df["InvoiceDate"].dt.hour
        return (
            df.groupby("Hour")
            .agg(
                Transactions=("InvoiceNo", "nunique"),
                Revenue=("Revenue", "sum")
            )
            .reset_index()
        )
class CustomerSegmentationStrategy(AnalyticsStrategy):
    """Group customers by purchasing behaviour."""
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.dropna(subset=["CustomerID"]).copy()
        return (
            df.groupby("CustomerID")
            .agg(
                PurchaseFrequency=("InvoiceNo", "nunique"),
                TotalSpending=("Revenue", "sum")
            )
            .reset_index()
        )

class SeasonalPromotionStrategy(AnalyticsStrategy):
    """Summarise sales by month."""
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.copy()
        df["InvoiceDate"] = pd.to_datetime(
            df["InvoiceDate"], errors="coerce"
        )
        df = df.dropna(subset=["InvoiceDate"])
        df["Month"] = df["InvoiceDate"].dt.month

        return (
            df.groupby("Month")
            .agg(
                Transactions=("InvoiceNo", "nunique"),
                Revenue=("Revenue", "sum")
            )
            .reset_index()
        )
class ProductPerformanceStrategy(AnalyticsStrategy):
    """Analyse product sales and revenue."""
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        return (
            data.groupby("Description", dropna=False)
            .agg(
                UnitsSold=("Quantity", "sum"),
                Revenue=("Revenue", "sum")
            )
            .reset_index()
            .sort_values("Revenue", ascending=False)
        )
class AnomalyDetectionStrategy(AnalyticsStrategy):
    """Identify unusually high or low transaction values."""
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.copy()
        q1 = df["Revenue"].quantile(0.25)
        q3 = df["Revenue"].quantile(0.75)
        iqr = q3 - q1
        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr
        return df[
            (df["Revenue"] < lower_bound)
            | (df["Revenue"] > upper_bound)
        ].copy()
class RegionalPerformanceStrategy(AnalyticsStrategy):
    """Compare revenue across regions."""
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        df = data.copy()
        if "Region" not in df.columns:
            raise ValueError(
                "The dataset must contain a Region column."
            )
        df["Region"] = (
            df["Region"].astype("string").str.strip().str.title()
        )
        return (
            df.groupby("Region", dropna=False)
            .agg(
                Transactions=("InvoiceNo", "nunique"),
                Revenue=("Revenue", "sum")
            )
            .reset_index()
            .sort_values("Revenue", ascending=False)
        )

In [18]:
import logging
from pathlib import Path
import pandas as pd
class OutputManager:
    """Manage analysis exports, rejected records and application logs."""
    def __init__(
        self,
        project_root: str | Path | None = None
    ):
        if project_root is None:
            project_root = Path(__file__).resolve().parent.parent
        self.root = Path(project_root)
        self.output_dir = self.root / "outputs"
        self.log_dir = self.root / "logs"
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.log_dir.mkdir(parents=True, exist_ok=True)
        self.logger = logging.getLogger(
            f"SmartRetailAnalytics.{self.root.resolve()}"
        )
        self.logger.setLevel(logging.INFO)
        if not self.logger.handlers:
            handler = logging.FileHandler(
                self.log_dir / "application.log",
                encoding="utf-8",
            )
            handler.setFormatter(
                logging.Formatter(
                    "%(asctime)s - %(levelname)s - %(message)s"
                )
            )
            self.logger.addHandler(handler)
    def save_analysis(
        self,
        data: pd.DataFrame,
        analysis_name: str,
    ) -> Path:
        """Export analytical results as a CSV file."""
        safe_name = "".join(
            char if char.isalnum() or char in "_-" else "_"
            for char in analysis_name
        )
        output_path = self.output_dir / f"{safe_name}.csv"
        data.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig",
        )
        self.logger.info(
            "Saved analysis '%s' with %d rows.",
            analysis_name,
            len(data),
        )
        return output_path

In [19]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd


class OutputManager:
    """Save SmartMart reports, rejected records, summaries and charts."""

    def __init__(self, output_dir: str):
        self.output_dir = Path(output_dir)
        self.charts_dir = self.output_dir / "charts"

        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.charts_dir.mkdir(parents=True, exist_ok=True)

    def save_report(
        self,
        data: pd.DataFrame,
        report_name: str
    ) -> Path:
        """Save an analytical report as an Excel file."""

        safe_name = self._safe_filename(report_name)
        file_path = self.output_dir / f"{safe_name}.xlsx"

        data.to_excel(file_path, index=False, engine="openpyxl")
        return file_path

    def save_csv(
        self,
        data: pd.DataFrame,
        report_name: str
    ) -> Path:
        """Save a report as a CSV file."""

        safe_name = self._safe_filename(report_name)
        file_path = self.output_dir / f"{safe_name}.csv"

        data.to_csv(
            file_path,
            index=False,
            encoding="utf-8-sig"
        )
        return file_path

    def save_rejected_records(
        self,
        data: pd.DataFrame
    ) -> Path:
        """Save rejected records with a timestamp."""

        timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
        file_path = (
            self.output_dir / f"invalid_records_{timestamp}.xlsx"
        )

        data.to_excel(file_path, index=False, engine="openpyxl")
        return file_path

    def save_dead_letter_queue(
        self,
        data: pd.DataFrame
    ) -> Path:
        """Save rejected records for later investigation."""

        file_path = self.output_dir / "dead_letter_queue.csv"

        data.to_csv(
            file_path,
            index=False,
            encoding="utf-8-sig"
        )
        return file_path

    def save_summary(
        self,
        summary: dict,
        summary_name: str = "processing_summary"
    ) -> Path:
        """Save processing statistics in JSON format."""

        timestamp = datetime.now().strftime("%Y%m%dT%H%M%S")
        safe_name = self._safe_filename(summary_name)

        file_path = (
            self.output_dir / f"{safe_name}_{timestamp}.json"
        )

        with file_path.open("w", encoding="utf-8") as file:
            json.dump(
                summary,
                file,
                indent=4,
                ensure_ascii=False,
                default=str
            )

        return file_path

    def save_chart(self, figure, chart_name: str) -> Path:
        """Save a Matplotlib chart as a PNG image."""

        safe_name = self._safe_filename(chart_name)
        file_path = self.charts_dir / f"{safe_name}.png"

        figure.savefig(
            file_path,
            dpi=300,
            bbox_inches="tight"
        )

        return file_path

    @staticmethod
    def _safe_filename(name: str) -> str:
        """Remove characters unsuitable for filenames."""

        safe_name = "".join(
            char if char.isalnum() or char in "_-" else "_"
            for char in name.strip()
        )

        return safe_name or "report"


In [21]:
import numpy as np
import pandas as pd
class DataValidator:
    """Validate retail transaction records and identify data-quality issues."""

    REQUIRED_COLUMNS = {
        "transaction_id",
        "transaction_datetime",
        "store_id",
        "product_category",
        "product_name",
        "quantity",
        "unit_price",
        "discount_percent",
        "total_amount",
        "transaction_type",
        "region",
        "store_city",
        "promotion_name",
        "customer_id",
    }

    NUMERIC_COLUMNS = [
        "quantity",
        "unit_price",
        "discount_percent",
        "total_amount",
    ]

    def validate(self, data: pd.DataFrame):
        """Separate valid and invalid records."""

        if data.empty:
            return data.copy(), data.copy()

        frame = data.copy()

        # Standardise column names
        frame.columns = [
            str(column).strip().lower()
            for column in frame.columns
        ]

        # Check required columns
        missing_columns = self.REQUIRED_COLUMNS - set(frame.columns)

        if missing_columns:
            raise ValueError(
                f"Missing required columns: {sorted(missing_columns)}"
            )

        # Remove duplicate records
        duplicate_mask = frame.duplicated()
        frame = frame.loc[~duplicate_mask].copy()

        # Convert numeric values
        for column in self.NUMERIC_COLUMNS:
            frame[column] = pd.to_numeric(
                frame[column], errors="coerce"
            )

        # Convert transaction dates
        frame["transaction_datetime"] = pd.to_datetime(
            frame["transaction_datetime"],
            errors="coerce"
        )

        # Identify missing essential values
        essential_fields = [
            "transaction_id",
            "transaction_datetime",
            "store_id",
            "product_category",
            "product_name",
            "quantity",
            "unit_price",
            "discount_percent",
            "total_amount",
            "transaction_type",
            "region",
            "store_city",
        ]

        invalid_mask = frame[essential_fields].isna().any(axis=1)

        # Validate numeric values
        numeric_values = frame[self.NUMERIC_COLUMNS].to_numpy(
            dtype=float
        )

        invalid_mask |= ~np.isfinite(numeric_values).all(axis=1)
        invalid_mask |= frame["unit_price"] < 0
        invalid_mask |= (
            (frame["discount_percent"] < 0)
            | (frame["discount_percent"] > 100)
        )

        # Validate transaction type
        transaction_type = (
            frame["transaction_type"]
            .astype("string")
            .str.strip()
            .str.casefold()
        )

        sale_mask = transaction_type.eq("sale")
        return_mask = transaction_type.eq("return")

        invalid_mask |= ~(sale_mask | return_mask)

        # Check quantity and amount signs
        invalid_mask |= sale_mask & (
            (frame["quantity"] < 0)
            | (frame["total_amount"] < 0)
        )

        invalid_mask |= return_mask & (
            (frame["quantity"] > 0)
            | (frame["total_amount"] > 0)
        )

        # Return separate DataFrames
        valid_records = frame.loc[~invalid_mask].copy()
        invalid_records = frame.loc[invalid_mask].copy()

        return valid_records, invalid_records


In [23]:
from abc import ABC, abstractmethod
import pandas as pd


# Common interface for all analytics strategies
class AnalyticsStrategy(ABC):
    name = "Analytics"

    @abstractmethod
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        """Perform a specific retail analysis."""
        pass


# Existing strategy
class PeakSalesStrategy(AnalyticsStrategy):
    name = "Peak Sales Analysis"

    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        result = data.copy()

        result["transaction_datetime"] = pd.to_datetime(
            result["transaction_datetime"],
            errors="coerce"
        )

        result = result.dropna(
            subset=["transaction_datetime"]
        )

        result = result[
            result["transaction_type"]
            .astype("string")
            .str.casefold()
            == "sale"
        ]

        result["hour"] = (
            result["transaction_datetime"].dt.hour
        )

        return (
            result.groupby("hour", as_index=False)
            .agg(
                transactions=("transaction_id", "count"),
                revenue=("total_amount", "sum")
            )
            .sort_values("revenue", ascending=False)
        )


# New strategy added without changing PeakSalesStrategy
class InventoryAnalysisStrategy(AnalyticsStrategy):
    name = "Inventory Analysis"

    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        return (
            data.groupby(
                ["product_category", "product_name"],
                as_index=False
            )
            .agg(
                units_sold=("quantity", "sum")
            )
            .sort_values("units_sold", ascending=False)
        )


# Engine that executes a selected strategy
class AnalyticsEngine:
    def __init__(self):
        self._strategies = {
            "peak_sales": PeakSalesStrategy(),
            "inventory": InventoryAnalysisStrategy(),
        }

    def run(
        self,
        analysis_name: str,
        data: pd.DataFrame
    ) -> pd.DataFrame:
        strategy = self._strategies.get(analysis_name)

        if strategy is None:
            raise ValueError(
                f"Unknown analysis: {analysis_name}"
            )

        return strategy.analyse(data)

In [25]:

from abc import ABC, abstractmethod
import pandas as pd


class AnalyticsStrategy(ABC):
    """Common interface for retail analytical strategies."""

    @abstractmethod
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        pass


class PeakSalesStrategy(AnalyticsStrategy):
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        result = data.copy()

        result["transaction_datetime"] = pd.to_datetime(
            result["transaction_datetime"],
            errors="coerce"
        )

        result = result.dropna(subset=["transaction_datetime"])
        result = result[
            result["transaction_type"].astype("string").str.casefold()
            == "sale"
        ]

        result["hour"] = result["transaction_datetime"].dt.hour

        return result.groupby("hour", as_index=False).agg(
            transactions=("transaction_id", "count"),
            revenue=("total_amount", "sum")
        )


class RegionalPerformanceStrategy(AnalyticsStrategy):
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        result = data.copy()

        result = result[
            result["transaction_type"].astype("string").str.casefold()
            == "sale"
        ]

        return result.groupby("region", as_index=False).agg(
            transactions=("transaction_id", "count"),
            revenue=("total_amount", "sum")
        )


def run_analysis(
    strategy: AnalyticsStrategy,
    data: pd.DataFrame
) -> pd.DataFrame:
    """Execute any strategy that follows the common interface."""
    return strategy.analyse(data)



In [28]:
from abc import ABC, abstractmethod
import pandas as pd
class AnalyticsStrategy(ABC):
    """A focused interface for individual analytics operations."""
    @abstractmethod
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        pass
class PeakSalesStrategy(AnalyticsStrategy):
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        result = data.copy()

        result["transaction_datetime"] = pd.to_datetime(
            result["transaction_datetime"],
            errors="coerce"
        )
        result = result.dropna(subset=["transaction_datetime"])
        result["hour"] = result["transaction_datetime"].dt.hour

        return result.groupby("hour", as_index=False).agg(
            revenue=("total_amount", "sum")
        )

class RegionalPerformanceStrategy(AnalyticsStrategy):
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        return data.groupby("region", as_index=False).agg(
            revenue=("total_amount", "sum")
        )


In [30]:

from abc import ABC, abstractmethod
import pandas as pd


class AnalyticsStrategy(ABC):
    """Abstraction for analytical operations."""

    @abstractmethod
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        pass


class PeakSalesStrategy(AnalyticsStrategy):
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        result = data.copy()

        result["transaction_datetime"] = pd.to_datetime(
            result["transaction_datetime"],
            errors="coerce"
        )

        result = result.dropna(subset=["transaction_datetime"])
        result["hour"] = result["transaction_datetime"].dt.hour

        return result.groupby("hour", as_index=False).agg(
            revenue=("total_amount", "sum")
        )


class AnalyticsEngine:
    """High-level component that receives an abstraction."""

    def __init__(self, strategy: AnalyticsStrategy):
        self.strategy = strategy

    def run(self, data: pd.DataFrame) -> pd.DataFrame:
        return self.strategy.analyse(data)



In [31]:

from pathlib import Path

# Run this script from your SmartMart project root.
src_dir = Path("src")

files = [
    "__init__.py",
    "analytics.py",
    "app_backup.py",
    "app.py",
    "charts.py",
    "data_reader.py",
    "data_validator.py",
    "main.py",
    "output_manager.py",
    "processing_service.py",
    "transformations.py",
]

src_dir.mkdir(parents=True, exist_ok=True)

for filename in files:
    file_path = src_dir / filename

    if file_path.exists():
        print(f"Already exists: {file_path}")
    else:
        file_path.touch()
        print(f"Created: {file_path}")

print("\nFolder structure created successfully.")


Created: src\__init__.py
Created: src\analytics.py
Created: src\app_backup.py
Created: src\app.py
Created: src\charts.py
Created: src\data_reader.py
Created: src\data_validator.py
Created: src\main.py
Created: src\output_manager.py
Created: src\processing_service.py
Created: src\transformations.py

Folder structure created successfully.


In [ ]:

import pandas as pd


class DataValidator:
    """Validate retail transaction records."""

    REQUIRED_COLUMNS = {
        "transaction_id",
        "transaction_datetime",
        "quantity",
        "unit_price",
        "discount_percent",
        "total_amount",
        "transaction_type",
    }

    NUMERIC_COLUMNS = [
        "quantity",
        "unit_price",
        "discount_percent",
        "total_amount",
    ]

    def validate(self, data: pd.DataFrame):
        if data.empty:
            return data.copy(), data.copy()

        frame = data.copy()

        # Standardise column names.
        frame.columns = [
            str(column).strip().lower()
            for column in frame.columns
        ]

        # Check required columns.
        missing_columns = self.REQUIRED_COLUMNS - set(frame.columns)

        if missing_columns:
            raise ValueError(
                "Missing required columns: "
                + ", ".join(sorted(missing_columns))
            )

        # Identify duplicate records.
        duplicate_mask = frame.duplicated()

        frame = frame.loc[~duplicate_mask].copy()

        # Convert numeric fields safely.
        for column in self.NUMERIC_COLUMNS:
            frame[column] = pd.to_numeric(
                frame[column],
                errors="coerce"
            )

        # Convert transaction dates.
        frame["transaction_datetime"] = pd.to_datetime(
            frame["transaction_datetime"],
            errors="coerce"
        )

        # Identify invalid records.
        invalid_mask = frame[
            [
                "transaction_id",
                "transaction_datetime",
                "quantity",
                "unit_price",
                "discount_percent",
                "total_amount",
                "transaction_type",
            ]
        ].isna().any(axis=1)

        invalid_mask |= frame["unit_price"] < 0
        invalid_mask |= (
            (frame["discount_percent"] < 0)
            | (frame["discount_percent"] > 100)
        )

        transaction_type = (
            frame["transaction_type"]
            .astype("string")
            .str.strip()
            .str.casefold()
        )

        invalid_mask |= ~transaction_type.isin(["sale", "return"])

        valid_records = frame.loc[~invalid_mask].copy()
        invalid_records = frame.loc[invalid_mask].copy()

        return valid_records, invalid_records


# Example usage
# validator = DataValidator()
# valid_data, invalid_data = validator.validate(dataset)


In [32]:

from abc import ABC, abstractmethod
import pandas as pd


class AnalyticsStrategy(ABC):
    """Common interface for retail analytics strategies."""

    name = "Analytics"

    @abstractmethod
    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        """Perform an analytical operation."""
        raise NotImplementedError


class PeakSalesStrategy(AnalyticsStrategy):
    """Analyse sales by transaction hour and day."""

    name = "Peak Sales Analysis"

    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        result = data.copy()

        result["transaction_datetime"] = pd.to_datetime(
            result["transaction_datetime"],
            errors="coerce"
        )

        result = result[
            result["transaction_type"]
            .astype("string")
            .str.casefold()
            == "sale"
        ]

        result = result.dropna(
            subset=["transaction_datetime"]
        )

        result["hour"] = (
            result["transaction_datetime"].dt.hour
        )

        result["day"] = (
            result["transaction_datetime"].dt.day_name()
        )

        return (
            result.groupby(
                ["hour", "day"],
                as_index=False
            )
            .agg(
                transactions=("transaction_id", "count"),
                revenue=("total_amount", "sum")
            )
            .sort_values("revenue", ascending=False)
        )


class RegionalPerformanceStrategy(AnalyticsStrategy):
    """Analyse retail sales by region."""

    name = "Regional Performance Analysis"

    def analyse(self, data: pd.DataFrame) -> pd.DataFrame:
        result = data.copy()

        result = result[
            result["transaction_type"]
            .astype("string")
            .str.casefold()
            == "sale"
        ]

        return (
            result.groupby("region", as_index=False)
            .agg(
                transactions=("transaction_id", "count"),
                revenue=("total_amount", "sum")
            )
            .sort_values("revenue", ascending=False)
        )


class AnalyticsEngine:
    """Execute different strategies through one interface."""

    def __init__(self):
        self.strategies = {
            "peak_sales": PeakSalesStrategy(),
            "regional_performance": RegionalPerformanceStrategy()
        }

    def run(
        self,
        analysis_name: str,
        data: pd.DataFrame
    ) -> pd.DataFrame:
        strategy = self.strategies.get(analysis_name)

        if strategy is None:
            raise ValueError(
                f"Unknown analysis: {analysis_name}"
            )

        return strategy.analyse(data)


In [1]:
import pytest
import pandas as pd

def calculate_sales(quantity, price):
    return quantity * price

def test_sales_calculation():
    assert calculate_sales(2, 100) == 200

def test_zero_quantity():
    assert calculate_sales(0, 100) == 0

def test_large_quantity():
    assert calculate_sales(100, 25) == 2500

print("Test functions defined successfully.")

Test functions defined successfully.


In [2]:
%%writefile test_retail.py
def calculate_sales(quantity, price):
    return quantity * price

def test_sales_calculation():
    assert calculate_sales(2, 100) == 200

def test_zero_quantity():
    assert calculate_sales(0, 100) == 0

def test_large_quantity():
    assert calculate_sales(100, 25) == 2500

Writing test_retail.py


In [3]:
!python -m pytest test_retail.py -v

============================= test session starts =============================
platform win32 -- Python 3.14.1, pytest-9.1.1, pluggy-1.7.0 -- C:\Users\USER'\AppData\Local\Programs\Python\Python314\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\USER'\OneDrive\Documents\APDP
plugins: anyio-4.13.0
collecting ... collected 3 items

test_retail.py::test_sales_calculation PASSED                            [ 33%]
test_retail.py::test_zero_quantity PASSED                                [ 66%]
test_retail.py::test_large_quantity PASSED                               [100%]

============================== 3 passed in 0.02s ==============================


In [6]:
def calculate_sales(quantity, price):
    return quantity * price

def test_normal_sales_calculation():
    assert calculate_sales(2, 100) == 200

def test_zero_quantity():
    assert calculate_sales(0, 100) == 0

def test_large_quantity():
    assert calculate_sales(100, 25) == 2500

In [5]:
import pytest


def calculate_valid_sales(quantity, price):
    if quantity < 0:
        raise ValueError("Quantity cannot be negative")

    if price < 0:
        raise ValueError("Price cannot be negative")

    return quantity * price


def test_negative_quantity():
    with pytest.raises(ValueError):
        calculate_valid_sales(-1, 100)


def test_negative_price():
    with pytest.raises(ValueError):
        calculate_valid_sales(2, -10)

In [10]:
import pytest
@pytest.mark.parametrize(
    "quantity, price, expected",
    [
        (0, 100, 0),
        (1, 100, 100),
        (100, 25, 2500),
    ]
)
def test_sales_boundaries(quantity, price, expected):
    assert quantity * price == pytest.approx(expected)

In [11]:
import pytest
def calculate_valid_sales(quantity, price):
    if quantity < 0:
        raise ValueError("Quantity cannot be negative")
    if price < 0:
        raise ValueError("Price cannot be negative")
    return quantity * price
def test_negative_quantity():
    with pytest.raises(ValueError):
        calculate_valid_sales(-1, 100)
def test_negative_price():
    with pytest.raises(ValueError):
        calculate_valid_sales(2, -10)

In [12]:
def test_retail_processing_pipeline():
    # Replace these placeholders with the actual
    # application components and methods.
    raw_data = load_test_transactions()

    validated_data = validate_transactions(raw_data)

    transformed_data = transform_transactions(
        validated_data
    )

    report = generate_sales_report(transformed_data)

    assert transformed_data is not None
    assert report is not None


In [13]:
def binary_search(items, target):
    low = 0
    high = len(items) - 1
    while low <= high:
        middle = (low + high) // 2
        if items[middle] == target:
            return middle
        if items[middle] < target:
            low = middle + 1
        else:
            high = middle - 1
    return -1
def test_target_in_middle():
    assert binary_search([10, 20, 30, 40, 50], 30) == 2
def test_target_at_start():
    assert binary_search([10, 20, 30, 40, 50], 10) == 0
def test_target_at_end():
    assert binary_search([10, 20, 30, 40, 50], 50) == 4
def test_target_not_found():
    assert binary_search([10, 20, 30, 40, 50], 35) == -1
def test_empty_list():
    assert binary_search([], 10) == -1

In [14]:
%%writefile test_binary_search.py
def binary_search(items, target):
    low = 0
    high = len(items) - 1
    while low <= high:
        middle = (low + high) // 2
        if items[middle] == target:
            return middle
        if items[middle] < target:
            low = middle + 1
        else:
            high = middle - 1
    return -1
def test_target_in_middle():
    assert binary_search([10, 20, 30, 40, 50], 30) == 2
def test_target_at_start():
    assert binary_search([10, 20, 30, 40, 50], 10) == 0
def test_target_at_end():
    assert binary_search([10, 20, 30, 40, 50], 50) == 4
def test_target_not_found():
    assert binary_search([10, 20, 30, 40, 50], 35) == -1
def test_empty_list():
    assert binary_search([], 10) == -1

Writing test_binary_search.py


In [15]:
import os
print(os.getcwd())

C:\Users\USER'\OneDrive\Documents\APDP


In [16]:
!python -m pytest test_binary_search.py -v

============================= test session starts =============================
platform win32 -- Python 3.14.1, pytest-9.1.1, pluggy-1.7.0 -- C:\Users\USER'\AppData\Local\Programs\Python\Python314\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\USER'\OneDrive\Documents\APDP
plugins: anyio-4.13.0
collecting ... collected 5 items

test_binary_search.py::test_target_in_middle PASSED                      [ 20%]
test_binary_search.py::test_target_at_start PASSED                       [ 40%]
test_binary_search.py::test_target_at_end PASSED                         [ 60%]
test_binary_search.py::test_target_not_found PASSED                      [ 80%]
test_binary_search.py::test_empty_list PASSED                            [100%]

============================== 5 passed in 0.02s ==============================


In [1]:
from pathlib import Path
import subprocess
import sys

# Create one pytest file automatically
test_code = r'''
import pytest
import pandas as pd


# 1. Sales calculation
def calculate_sales(quantity, price):
    return quantity * price


def test_normal_sales_calculation():
    assert calculate_sales(2, 100) == 200


def test_zero_quantity():
    assert calculate_sales(0, 100) == 0


def test_large_quantity():
    assert calculate_sales(100, 25) == 2500


# 2. Parameterized boundary-value testing
@pytest.mark.parametrize(
    "quantity, price, expected",
    [
        (0, 100, 0),
        (1, 100, 100),
        (100, 25, 2500),
    ]
)
def test_sales_boundaries(quantity, price, expected):
    assert calculate_sales(quantity, price) == pytest.approx(expected)


# 3. Negative input and exception testing
def calculate_valid_sales(quantity, price):
    if quantity < 0:
        raise ValueError("Quantity cannot be negative")
    if price < 0:
        raise ValueError("Price cannot be negative")
    return quantity * price


def test_negative_quantity():
    with pytest.raises(ValueError):
        calculate_valid_sales(-1, 100)


def test_negative_price():
    with pytest.raises(ValueError):
        calculate_valid_sales(2, -10)


# 4. Binary search algorithm testing
def binary_search(items, target):
    low = 0
    high = len(items) - 1

    while low <= high:
        middle = (low + high) // 2

        if items[middle] == target:
            return middle
        elif items[middle] < target:
            low = middle + 1
        else:
            high = middle - 1

    return -1


def test_binary_search_middle():
    assert binary_search([10, 20, 30, 40, 50], 30) == 2


def test_binary_search_start():
    assert binary_search([10, 20, 30, 40, 50], 10) == 0


def test_binary_search_end():
    assert binary_search([10, 20, 30, 40, 50], 50) == 4


def test_binary_search_not_found():
    assert binary_search([10, 20, 30, 40, 50], 35) == -1


def test_binary_search_empty_list():
    assert binary_search([], 10) == -1


# 5. CSV reading and data checking
def test_csv_file_reading(tmp_path):
    csv_file = tmp_path / "retail_sales.csv"

    expected = pd.DataFrame({
        "Product": ["Laptop", "Mouse"],
        "Quantity": [2, 5],
        "Price": [1000, 20]
    })

    expected.to_csv(csv_file, index=False)
    actual = pd.read_csv(csv_file)

    assert len(actual) == 2
    assert list(actual.columns) == ["Product", "Quantity", "Price"]
    assert actual["Quantity"].sum() == 7
'''

# Save the test file in the notebook's current directory
test_file = Path.cwd() / "test_task4_pytest.py"
test_file.write_text(test_code, encoding="utf-8")

print("Pytest file created:", test_file)
print("\nRunning automated tests...\n")

# Run pytest using the same Python environment as Jupyter
result = subprocess.run(
    [
        sys.executable, "-m", "pytest",
        str(test_file), "-v", "--tb=short"
    ],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)

print("Pytest exit code:", result.returncode)


Pytest file created: C:\Users\USER'\OneDrive\Documents\APDP\test_task4_pytest.py

Running automated tests...

============================= test session starts =============================
platform win32 -- Python 3.14.1, pytest-9.1.1, pluggy-1.7.0 -- C:\Users\USER'\AppData\Local\Programs\Python\Python314\python.exe
cachedir: .pytest_cache
rootdir: C:\Users\USER'\OneDrive\Documents\APDP
plugins: anyio-4.13.0
collecting ... collected 14 items

test_task4_pytest.py::test_normal_sales_calculation PASSED               [  7%]
test_task4_pytest.py::test_zero_quantity PASSED                          [ 14%]
test_task4_pytest.py::test_large_quantity PASSED                         [ 21%]
test_task4_pytest.py::test_sales_boundaries[0-100-0] PASSED              [ 28%]
test_task4_pytest.py::test_sales_boundaries[1-100-100] PASSED            [ 35%]
test_task4_pytest.py::test_sales_boundaries[100-25-2500] PASSED          [ 42%]
test_task4_pytest.py::test_negative_quantity PASSED                     

In [2]:

from pathlib import Path

test_code = r'''
import pytest
import pandas as pd


# 1. Unit testing: sales calculation
def calculate_sales(quantity, price):
    return quantity * price


def test_normal_sales_calculation():
    assert calculate_sales(2, 100) == 200


def test_zero_quantity():
    assert calculate_sales(0, 100) == 0


def test_large_quantity():
    assert calculate_sales(100, 25) == 2500


# 2. Parameterized boundary-value testing
@pytest.mark.parametrize(
    "quantity, price, expected",
    [
        (0, 100, 0),
        (1, 100, 100),
        (100, 25, 2500),
    ]
)
def test_sales_boundaries(quantity, price, expected):
    assert calculate_sales(quantity, price) == pytest.approx(expected)


# 3. Negative testing and exception handling
def calculate_valid_sales(quantity, price):
    if quantity < 0:
        raise ValueError("Quantity cannot be negative")
    if price < 0:
        raise ValueError("Price cannot be negative")
    return quantity * price


def test_negative_quantity():
    with pytest.raises(ValueError):
        calculate_valid_sales(-1, 100)


def test_negative_price():
    with pytest.raises(ValueError):
        calculate_valid_sales(2, -10)


# 4. Binary search algorithm testing
def binary_search(items, target):
    low = 0
    high = len(items) - 1

    while low <= high:
        middle = (low + high) // 2

        if items[middle] == target:
            return middle
        elif items[middle] < target:
            low = middle + 1
        else:
            high = middle - 1

    return -1


def test_binary_search_middle():
    assert binary_search([10, 20, 30, 40, 50], 30) == 2


def test_binary_search_start():
    assert binary_search([10, 20, 30, 40, 50], 10) == 0


def test_binary_search_end():
    assert binary_search([10, 20, 30, 40, 50], 50) == 4


def test_binary_search_not_found():
    assert binary_search([10, 20, 30, 40, 50], 35) == -1


def test_binary_search_empty_list():
    assert binary_search([], 10) == -1


# 5. CSV file handling and data checks
def test_csv_file_reading(tmp_path):
    csv_file = tmp_path / "retail_sales.csv"

    expected = pd.DataFrame({
        "Product": ["Laptop", "Mouse"],
        "Quantity": [2, 5],
        "Price": [1000, 20]
    })

    expected.to_csv(csv_file, index=False)
    actual = pd.read_csv(csv_file)

    assert len(actual) == 2
    assert list(actual.columns) == [
        "Product", "Quantity", "Price"
    ]
    assert actual["Quantity"].sum() == 7
'''

test_file = Path.cwd() / "test_task4_pytest.py"
test_file.write_text(test_code, encoding="utf-8")

print("Test file created successfully:")
print(test_file)
print("\nYou can now run this file from the terminal.")


Test file created successfully:
C:\Users\USER'\OneDrive\Documents\APDP\test_task4_pytest.py

You can now run this file from the terminal.


In [4]:
from pathlib import Path
import pandas as pd
import pytest


# ---------------------------------------------------------
# TEST FIXTURES
# ---------------------------------------------------------

@pytest.fixture
def valid_retail_df():
    """Provide a small, valid retail dataset for testing."""
    return pd.DataFrame({
        "TransactionID": ["T001", "T002", "T003"],
        "Product": ["Laptop", "Mouse", "Keyboard"],
        "Quantity": [2, 5, 3],
        "Price": [1000.00, 20.00, 35.00],
        "Region": ["North", "South", "East"],
    })


@pytest.fixture
def empty_df():
    """Provide an empty DataFrame."""
    return pd.DataFrame()


@pytest.fixture
def temporary_output_dir(tmp_path):
    """Provide a temporary directory for test outputs."""
    return tmp_path


# ---------------------------------------------------------
# CONNECTION TO THE REAL APPLICATION
# ---------------------------------------------------------

@pytest.fixture
def app_api():
    """
    Connect this fixture to your actual SmartRetail modules.

    Replace the NotImplementedError with imports and calls
    to your real CSV reader, validator, transformations,
    analytics engine, report generator, and processing service.
    """
    raise NotImplementedError(
        "Connect app_api to the actual SmartRetail implementation."
    )


# ---------------------------------------------------------
# TC01: GUI DATASET PROCESSING
# ---------------------------------------------------------

def test_tc01_gui_dataset_processing(valid_retail_df):
    """Verify the sample dataset is suitable for processing."""
    assert not valid_retail_df.empty
    assert len(valid_retail_df) == 3
    assert {"Quantity", "Price", "Product"}.issubset(
        valid_retail_df.columns
    )


# ---------------------------------------------------------
# TC02: EMPTY CSV FILE
# ---------------------------------------------------------

def test_tc02_empty_csv_file(tmp_path):
    """Verify that an empty CSV is identified."""
    csv_path = tmp_path / "empty.csv"
    csv_path.write_text("", encoding="utf-8")

    assert csv_path.exists()
    assert csv_path.stat().st_size == 0

    # pandas raises EmptyDataError for a zero-byte CSV.
    # Your application should catch or handle this condition.
    with pytest.raises(pd.errors.EmptyDataError):
        pd.read_csv(csv_path)


# ---------------------------------------------------------
# TC03: AUTOMATED DATA VALIDATION
# ---------------------------------------------------------

def test_tc03_required_columns(valid_retail_df):
    """Check that required columns are present."""
    required_columns = {"Product", "Quantity", "Price"}

    assert required_columns.issubset(valid_retail_df.columns)


def test_tc03_missing_required_column(valid_retail_df):
    """Detect a missing required column."""
    invalid_df = valid_retail_df.drop(columns=["Price"])
    required_columns = {"Product", "Quantity", "Price"}

    missing = required_columns - set(invalid_df.columns)

    assert missing == {"Price"}


# ---------------------------------------------------------
# TC04: DUPLICATE RECORD DETECTION
# ---------------------------------------------------------

def test_tc04_duplicate_record_detection(valid_retail_df):
    """Verify that a repeated row is recognised as a duplicate."""
    duplicated_df = pd.concat(
        [valid_retail_df, valid_retail_df.iloc[[0]]],
        ignore_index=True,
    )

    duplicate_mask = duplicated_df.duplicated()
    assert int(duplicate_mask.sum()) == 1


# ---------------------------------------------------------
# TC05: SIX ANALYTICAL STRATEGIES
# ---------------------------------------------------------

@pytest.mark.parametrize(
    "strategy_name",
    [
        "peak_sales",
        "customer_segmentation",
        "seasonal_promotion",
        "product_performance",
        "anomaly_detection",
        "regional_performance",
    ],
)
def test_tc05_strategy_names_are_unique(strategy_name):
    """
    Basic test of the six expected strategy identifiers.
    Replace with a test that invokes each real strategy.
    """
    expected_strategies = {
        "peak_sales",
        "customer_segmentation",
        "seasonal_promotion",
        "product_performance",
        "anomaly_detection",
        "regional_performance",
    }

    assert strategy_name in expected_strategies


# ---------------------------------------------------------
# TC06: REPORT GENERATION
# ---------------------------------------------------------

def test_tc06_output_directory_is_available(
    temporary_output_dir,
):
    """Verify that the temporary report directory can be used."""
    report_path = temporary_output_dir / "sales_report.csv"

    sample_report = pd.DataFrame({
        "Product": ["Laptop"],
        "TotalSales": [2000.00],
    })
    sample_report.to_csv(report_path, index=False)

    assert report_path.exists()
    assert pd.read_csv(report_path).loc[0, "TotalSales"] == 2000.00


# ---------------------------------------------------------
# TC07: CHART GENERATION
# ---------------------------------------------------------

def test_tc07_chart_output_path(temporary_output_dir):
    """Check that a chart file can be saved to the test directory."""
    chart_path = temporary_output_dir / "sales_chart.png"

    # A placeholder file is used here to test the output path only.
    # Replace this with the real chart-generation function.
    chart_path.write_bytes(b"test image placeholder")

    assert chart_path.exists()
    assert chart_path.stat().st_size > 0


# ---------------------------------------------------------
# TC08: INVALID RECORD OUTPUT
# ---------------------------------------------------------

def test_tc08_invalid_record_output(temporary_output_dir):
    """Verify invalid records can be exported to CSV."""
    invalid_records = pd.DataFrame({
        "TransactionID": ["T999"],
        "Quantity": [-5],
        "Price": [20.00],
        "Error": ["Quantity cannot be negative"],
    })

    output_path = temporary_output_dir / "invalid_records.csv"
    invalid_records.to_csv(output_path, index=False)

    saved_records = pd.read_csv(output_path)

    assert len(saved_records) == 1
    assert saved_records.loc[0, "Quantity"] == -5
    assert "Error" in saved_records.columns


# ---------------------------------------------------------
# TC09: EMPTY INPUT DATAFRAME
# ---------------------------------------------------------

def test_tc09_empty_dataframe(empty_df):
    """Verify that the test input is an empty DataFrame."""
    assert empty_df.empty
    assert len(empty_df) == 0


# ---------------------------------------------------------
# TC10: PARTIAL PROCESSING FAILURE
# ---------------------------------------------------------

def test_tc10_partial_processing_failure():
    """Verify pytest can assert a simulated processing exception."""

    def process_chunk(chunk_number):
        if chunk_number == 2:
            raise ValueError("Simulated chunk failure")
        return f"Chunk {chunk_number} processed"

    assert process_chunk(1) == "Chunk 1 processed"

    with pytest.raises(ValueError, match="Simulated chunk failure"):
        process_chunk(2)


# ---------------------------------------------------------
# TC11: COMPLETE PROCESSING FAILURE
# ---------------------------------------------------------

def test_tc11_complete_processing_failure():
    """Verify that a simulated fatal processing error is detectable."""

    def run_processing():
        raise RuntimeError("Simulated processing failure")

    with pytest.raises(RuntimeError, match="Simulated processing failure"):
        run_processing()


# ---------------------------------------------------------
# TC12: GUI RESULTS DISPLAY - LOGIC LEVEL
# ---------------------------------------------------------

def test_tc12_processing_summary(valid_retail_df):
    """
    Check the data used to prepare a GUI summary.
    This does not launch or test the actual GUI.
    """
    summary = {
        "records_processed": len(valid_retail_df),
        "status": "Completed",
    }

    assert summary["records_processed"] == 3
    assert summary["status"] == "Completed"


In [6]:

import pandas as pd
import pytest


@pytest.fixture
def retail_data():
    return pd.DataFrame({
        "TransactionID": ["T001", "T002", "T003"],
        "Product": ["Laptop", "Mouse", "Keyboard"],
        "Quantity": [2, 5, 3],
        "Price": [1000, 20, 35],
    })


def test_tc01_dataset_processing(retail_data):
    assert len(retail_data) == 3
    assert not retail_data.empty


def test_tc02_empty_csv(tmp_path):
    csv_file = tmp_path / "empty.csv"
    csv_file.write_text("", encoding="utf-8")

    assert csv_file.exists()
    assert csv_file.stat().st_size == 0


def test_tc03_data_validation(retail_data):
    required = {"Product", "Quantity", "Price"}
    assert required.issubset(retail_data.columns)


def test_tc04_duplicate_detection(retail_data):
    data = pd.concat(
        [retail_data, retail_data.iloc[[0]]],
        ignore_index=True,
    )
    assert data.duplicated().sum() == 1


@pytest.mark.parametrize(
    "quantity,price,expected",
    [(2, 100, 200), (0, 100, 0), (5, 20, 100)],
)
def test_tc05_sales_calculation(quantity, price, expected):
    assert quantity * price == pytest.approx(expected)


def test_tc06_report_generation(tmp_path):
    report = pd.DataFrame({"Product": ["Laptop"], "Sales": [2000]})
    path = tmp_path / "report.csv"
    report.to_csv(path, index=False)

    assert path.exists()
    assert pd.read_csv(path).loc[0, "Sales"] == 2000


def test_tc07_chart_output(tmp_path):
    path = tmp_path / "chart.png"
    path.write_bytes(b"test")

    assert path.exists()
    assert path.stat().st_size > 0


def test_tc08_invalid_record_output(tmp_path):
    invalid = pd.DataFrame({
        "Quantity": [-5],
        "Error": ["Invalid quantity"],
    })
    path = tmp_path / "invalid.csv"
    invalid.to_csv(path, index=False)

    saved = pd.read_csv(path)
    assert len(saved) == 1
    assert saved.loc[0, "Quantity"] == -5


def test_tc09_empty_dataframe():
    assert pd.DataFrame().empty


def test_tc10_partial_failure():
    with pytest.raises(ValueError):
        raise ValueError("Simulated partial failure")


def test_tc11_complete_failure():
    with pytest.raises(RuntimeError):
        raise RuntimeError("Simulated complete failure")


def test_tc12_results_summary(retail_data):
    summary = {
        "records_processed": len(retail_data),
        "status": "Completed",
    }

    assert summary["records_processed"] == 3
    assert summary["status"] == "Completed"


In [1]:
from pathlib import Path
import pandas as pd
import pytest


In [2]:
@pytest.fixture
def valid_retail_df():
    return pd.DataFrame({
        "TransactionID": ["T001", "T002", "T003", "T004"],
        "Product": ["Laptop", "Mouse", "Keyboard", "Laptop"],
        "Quantity": [2, 5, 3, 1],
        "Price": [1000.0, 20.0, 35.0, 1000.0],
        "Region": ["North", "South", "East", "North"],
        "CustomerID": ["C01", "C02", "C03", "C01"],
    })

In [3]:
def validate_dataframe(df, required):
    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    valid_mask = (df["Quantity"] >= 0) & (df["Price"] >= 0)
    return df.loc[valid_mask].copy(), df.loc[~valid_mask].copy()

In [4]:
def detect_duplicates(df):
    return df[df.duplicated(keep="first")].copy()

In [5]:
def calculate_sales(df):
    result = df.copy()
    result["TotalSales"] = result["Quantity"] * result["Price"]
    return result

In [6]:
@pytest.mark.parametrize("strategy", [
    "peak_sales",
    "customer_segmentation",
    "seasonal_promotion",
    "product_performance",
    "anomaly_detection",
    "regional_performance",
])
def test_tc05_each_analytics_strategy_returns_dataframe(
    strategy, valid_retail_df
):
    result = run_demo_strategy(strategy, valid_retail_df)
    assert isinstance(result, pd.DataFrame)
    assert not result.empty

In [7]:
def test_tc06_report_generation(tmp_path, valid_retail_df):
    report = calculate_sales(valid_retail_df)
    report_path = tmp_path / "sales_report.csv"
    write_report(report, report_path)

    assert report_path.exists()
    loaded = pd.read_csv(report_path)
    assert len(loaded) == len(valid_retail_df)
    assert "TotalSales" in loaded.columns

In [12]:
with pytest.raises(RuntimeError):
    failing_processing_service()

In [15]:
assert summary["records_processed"] == 4
assert summary["total_sales"] == pytest.approx(3205.0)
assert summary["status"] == "Completed"

In [11]:
import pytest

def failing_processing_service():
    raise RuntimeError("Simulated complete processing failure")

with pytest.raises(RuntimeError, match="Simulated complete processing failure"):
    failing_processing_service()

print("TC11 passed: Complete processing failure handled correctly.")

TC11 passed: Complete processing failure handled correctly.


In [14]:
import pandas as pd
import pytest

retail_data = pd.DataFrame({
    "Quantity": [2, 5, 3, 1],
    "Price": [1000.0, 20.0, 35.0, 1000.0],
})

processed = retail_data.copy()
processed["TotalSales"] = processed["Quantity"] * processed["Price"]

summary = {
    "records_processed": len(processed),
    "total_sales": float(processed["TotalSales"].sum()),
    "status": "Completed",
}

assert summary["records_processed"] == 4
assert summary["total_sales"] == pytest.approx(3205.0)
assert summary["status"] == "Completed"

print("TC12 passed: Results summary verified.")
print(summary)

TC12 passed: Results summary verified.
{'records_processed': 4, 'total_sales': 3205.0, 'status': 'Completed'}
